# Shape: profile a generated domain and check it against its contract (Synapse PySpark notebook)

Run after `shape_generate_synapse`. Reads every table named in the domain's contract (Delta or
Parquet folders under `tablesPath`), profiles them together on the driver (so keys and foreign keys
between tables are detected), checks the profile against the contract, optionally diffs it against
a baseline, writes `.shape`, `.html` and `.summary.json` below `outputPath/<domain>/<timestamp>/`,
and returns a compact JSON result to the calling pipeline.

The exit value is `{domain, tables, rows, passed, violations, drifted, changes, artifactPath,
truncated, kernel}`. `passed` is false when any table breaks the contract. `mssparkutils.notebook.exit`
is the last statement, outside any `try`/`except`.


In [ ]:
# Parameters cell (Synapse: toggle "parameter cell"). A pipeline overrides these.
# Paths are abfss://<container>@<account>.dfs.core.windows.net/<path> URLs.
domain = "retail"  # labels the artifacts
tablesPath = (
    "abfss://shape@<account>.dfs.core.windows.net/generated/tables"  # shape_generate_synapse's
)
tablePrefix = ""  # the tables are <tablesPath>/<tablePrefix><table>
tableFormat = "delta"  # delta | parquet
contractPath = "abfss://shape@<account>.dfs.core.windows.net/generated/retail/contract.json"
baselinePath = ""  # optional earlier .shape artifact (abfss URL) to diff against
outputPath = "abfss://shape@<account>.dfs.core.windows.net/shape"  # artifacts go below this
failOnDrift = False  # True: drift against the baseline also fails the gate
linkedServiceName = ""  # optional: Synapse linked service that grants access to the storage

In [ ]:
import json
import tempfile
from pathlib import Path

import pyarrow as pa

import shape
from shape.integrations.fabric import generation
from shape.integrations.fabric.run_folder import unique_run_name
from shape.kernel.dispatch import get_kernel

try:  # Synapse Spark pools predefine `mssparkutils`; newer runtimes also offer the import
    from notebookutils import mssparkutils
except ImportError:
    pass

KERNEL = get_kernel().NAME
print(f"Shape {shape.__version__}, kernel: {KERNEL}")


def _as_bool(value) -> bool:
    """Pipeline parameters may arrive as strings."""
    if isinstance(value, str):
        return value.strip().lower() in ("1", "true", "yes", "y")
    return bool(value)


failOnDrift = _as_bool(failOnDrift)
tableFormat = str(tableFormat).strip().lower()
if tableFormat not in ("delta", "parquet"):
    raise ValueError(f"tableFormat must be 'delta' or 'parquet', got {tableFormat!r}")
generation.check_name(str(domain), "domain")
if tablePrefix:
    generation.check_name(str(tablePrefix), "tablePrefix")

if linkedServiceName:
    spark.conf.set("spark.storage.synapse.linkedServiceName", linkedServiceName)  # noqa: F821
    spark.conf.set(  # noqa: F821
        "fs.azure.account.oauth.provider.type",
        "com.microsoft.azure.synapse.tokenlibrary.LinkedServiceBasedTokenProvider",
    )

LOCAL = Path(tempfile.mkdtemp(prefix="shape_"))


def _read_text(url: str) -> str:
    return mssparkutils.fs.head(url, 64 * 1024 * 1024)  # noqa: F821


def _fetch(url: str, name: str) -> str:
    local = LOCAL / name
    mssparkutils.fs.cp(url, f"file:{local}", False)  # noqa: F821
    return str(local)


def _publish_file(local: Path, url: str) -> None:
    mssparkutils.fs.cp(f"file:{local}", url, True)  # noqa: F821


def _publish_text(text: str, url: str) -> None:
    mssparkutils.fs.put(url, text, True)  # noqa: F821


safe_name = str(domain)
out_parent = f"{str(outputPath).rstrip('/')}/{safe_name}"  # one new folder per run below it
MAX_LISTED = 100  # keep the exit value small (well under 1 MB)
DRIVER_ROW_LIMIT = 20_000_000  # the Arrow copy of every table lives in driver memory

In [ ]:
contract = json.loads(_read_text(contractPath))

# The contract names the tables: every one of them is read and profiled, whole (a sample would
# fail the contract's row counts).
sources = {}
for name in contract["tables"]:
    generation.check_name(name, "table")
    df = spark.read.format(tableFormat).load(  # noqa: F821
        f"{str(tablesPath).rstrip('/')}/{tablePrefix}{name}"
    )
    if int(spark.version.split(".")[0]) >= 4:  # noqa: F821
        sources[name] = df.toArrow()
    else:
        sources[name] = pa.Table.from_pandas(df.toPandas(), preserve_index=False)
    if sum(t.num_rows for t in sources.values()) > DRIVER_ROW_LIMIT:
        raise ValueError(f"more than {DRIVER_ROW_LIMIT:,} rows: profile a smaller scale")
total_rows = sum(t.num_rows for t in sources.values())
profile = shape.profile(sources, name=str(domain))
print(f"Profiled {total_rows:,} rows in {len(sources)} tables")

In [ ]:
violations, changes, drifted = list(shape.check(profile, contract).violations), [], False

if baselinePath:
    diff_result = shape.diff(shape.load(_fetch(baselinePath, "baseline.shape")), profile)
    drifted = bool(diff_result.drifted)
    changes = list(diff_result.changes)
    if drifted and failOnDrift:
        violations.append(
            {
                "column": "*",
                "rule": "drift",
                "expected": "no drift against baseline",
                "observed": f"{len(changes)} change(s)",
            }
        )

passed = not violations

In [ ]:
out_root = f"{out_parent}/{unique_run_name(out_parent, mssparkutils.fs.exists)}"
local_shape = LOCAL / f"{safe_name}.shape"
shape.save(profile, str(local_shape))
artifact_path = f"{out_root}/{safe_name}.shape"
_publish_file(local_shape, artifact_path)
_publish_text(profile.to_html(), f"{out_root}/{safe_name}.html")
_publish_text(json.dumps(profile.summary()), f"{out_root}/{safe_name}.summary.json")
print("Artifacts written to", out_root)
displayHTML(profile.to_html())  # noqa: F821 (predefined in Synapse notebooks)

In [ ]:
result = {
    "domain": domain,
    "tables": {name: t.num_rows for name, t in sources.items()},
    "rows": total_rows,
    "passed": passed,
    "violations": violations[:MAX_LISTED],
    "drifted": drifted,
    "changes": changes[:MAX_LISTED],
    "artifactPath": artifact_path,
    "truncated": len(violations) > MAX_LISTED or len(changes) > MAX_LISTED,
    "kernel": KERNEL,
}
print(json.dumps(result, indent=2, default=str)[:4000])

In [ ]:
mssparkutils.notebook.exit(json.dumps(result, default=str))  # noqa: F821